# Pruning + Knowledge Distillation Pipeline for BERT

This notebook demonstrates a complete model compression pipeline:

1. **Unstructured Pruning** on a `bert-tiny` student model using Mase's L1-norm pruning pass.
2. **Knowledge Distillation (KD)** from a fine-tuned `bert-base` teacher to recover accuracy lost during pruning.

Three distillation strategies are explored progressively:

| Strategy | Description | Key Hyperparameters |
|----------|-------------|---------------------|
| **A** | Logits-only KD | `alpha_kd=1.0`, `T=3.0` |
| **B** | Logits + Hidden layer KD | `alpha_kd=2.0`, `alpha_hidden=1.0`, `T=5.0` |
| **C** | Logits + Hidden + Attention KD | `alpha_kd=1.0`, `alpha_hidden=0.1`, `alpha_attn=0.1`, `T=2.0` |

**Models used:**
- Student: `prajjwal1/bert-tiny` (2 layers, hidden_dim=128, ~4.4M params)
- Teacher: `textattack/bert-base-uncased-IMDB` (12 layers, hidden_dim=768, ~110M params)
- Dataset: IMDb sentiment classification

## 0. Configuration

In [ ]:
import sys
import os

# Ensure the custom mase_kd module is importable
sys.path.append(os.path.abspath("src"))

# Model and dataset identifiers
STUDENT_CHECKPOINT = "prajjwal1/bert-tiny"
TEACHER_CHECKPOINT = "textattack/bert-base-uncased-IMDB"
TOKENIZER_CHECKPOINT = "bert-base-uncased"
DATASET_NAME = "imdb"

# Architecture dimensions
STUDENT_DIM = 128
TEACHER_DIM = 768
STUDENT_LAYERS = 2
TEACHER_LAYERS = 12

NUM_TRAIN_EPOCHS = 5

## 1. Load the Student Model

Create a `MaseGraph` for `bert-tiny` from scratch, or load a previously saved QAT checkpoint.

In [ ]:
from transformers import AutoModelForSequenceClassification
from chop import MaseGraph
import chop.passes as passes

# Option A: Initialize from pretrained weights
model = AutoModelForSequenceClassification.from_pretrained(STUDENT_CHECKPOINT)
model.config.problem_type = "single_label_classification"

mg = MaseGraph(
    model,
    hf_input_names=["input_ids", "attention_mask", "labels"],
)

mg, _ = passes.init_metadata_analysis_pass(mg)
mg, _ = passes.add_common_metadata_analysis_pass(mg)

In [ ]:
# Option B: Load from a previously saved QAT checkpoint (Tutorial 3)
from pathlib import Path

mg = MaseGraph.from_checkpoint(f"{Path.home()}/tutorial_3_qat")

## 2. Prepare the Dataset

In [ ]:
from chop.tools import get_tokenized_dataset, get_trainer

dataset, tokenizer = get_tokenized_dataset(
    dataset=DATASET_NAME,
    checkpoint=TOKENIZER_CHECKPOINT,
    return_tokenizer=True,
)

## 3. Pre-Pruning Baseline Evaluation

Evaluate the student model before pruning. If coming from Tutorial 3 (QAT), this reflects post-QAT accuracy. Otherwise, it will likely be around 50% (random guess).

In [ ]:
trainer = get_trainer(
    model=mg.model,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
)

eval_results = trainer.evaluate()
print(f"Pre-pruning accuracy: {eval_results['eval_accuracy']}")

## 4. Unstructured Pruning

Apply L1-norm unstructured pruning at 50% sparsity with local scope on both weights and activations.

- **Sparsity**: proportion of elements set to zero (0.5 = 50%).
- **Method**: `l1-norm` prunes elements with the smallest absolute values.
- **Scope**: `local` computes thresholds per tensor; `global` uses a single threshold across all tensors.

In [ ]:
pruning_config = {
    "weight": {"sparsity": 0.5, "method": "l1-norm", "scope": "local"},
    "activation": {"sparsity": 0.5, "method": "l1-norm", "scope": "local"},
}

mg, _ = passes.prune_transform_pass(mg, pass_args=pruning_config)

### Post-Pruning Evaluation

Accuracy typically drops by ~10% or more after pruning.

In [ ]:
trainer = get_trainer(
    model=mg.model,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
    num_train_epochs=NUM_TRAIN_EPOCHS,
)

eval_results = trainer.evaluate()
print(f"Post-pruning accuracy: {eval_results['eval_accuracy']}")

## 5. Load the Teacher Model

Load a fine-tuned `bert-base-uncased` teacher model (trained on IMDB) and freeze it in eval mode.

In [ ]:
print(f"Loading teacher model from {TEACHER_CHECKPOINT}...")
teacher_model = AutoModelForSequenceClassification.from_pretrained(TEACHER_CHECKPOINT)
teacher_model.eval()
print("Teacher model loaded and set to eval mode.")

---
## 6. Strategy A: Logits-Only Knowledge Distillation

Use KL-divergence on softened logits from teacher and student to recover pruning-induced accuracy loss.

In [ ]:
from mase_kd.distillation import KnowledgeDistillationPass

# Wrap the pruned student with a logits-only KD objective
# Student (bert-tiny): 2 layers, dim=128
# Teacher (bert-base): 12 layers, dim=768
kd_model = KnowledgeDistillationPass(
    student_model=mg.model,
    teacher_model=teacher_model,
    s_dim=STUDENT_DIM,
    t_dim=TEACHER_DIM,
    s_layers=STUDENT_LAYERS,
    t_layers=TEACHER_LAYERS,
    alpha_kd=1.0,
    temperature=3.0,
)

print("KD wrapper initialized (Logits-Only strategy for FX-traced graph compatibility).")

### Strategy A: Training

In [ ]:
kd_trainer = get_trainer(
    model=kd_model,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
    num_train_epochs=NUM_TRAIN_EPOCHS,
)

print("Starting Logits-Only KD fine-tuning...")
kd_trainer.train()

### Strategy A: Evaluation

In [ ]:
# Evaluate the student model (without the KD wrapper) for a fair comparison
eval_trainer = get_trainer(
    model=mg.model,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
)

eval_results = eval_trainer.evaluate()
print(f"Strategy A accuracy (Logits-Only KD): {eval_results['eval_accuracy']}")

### Strategy A: Save Checkpoint

In [ ]:
import torch

save_dir = "saved_models/kd_logits_only"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, "student_logits_only.pt")

# Only save the student model weights (no need to store the large teacher)
torch.save(mg.model.state_dict(), save_path)
print(f"Strategy A weights saved to: {save_path}")

---
## 7. Strategy B: Logits + Hidden Layer Joint Distillation

In addition to logits KD, align intermediate hidden representations between student and teacher using a projection layer.

In [ ]:
from mase_kd.distillation.kd_pass_hidden import kd_transform_pass

kd_hidden_config = {
    "s_dim": STUDENT_DIM,
    "t_dim": TEACHER_DIM,
    "s_layers": STUDENT_LAYERS,
    "t_layers": TEACHER_LAYERS,
    "alpha_kd": 2.0,
    "alpha_hidden": 1.0,
    "temperature": 5.0,
}

kd_hidden_model, hidden_pass_info = kd_transform_pass(
    student_graph_or_model=mg,
    teacher_model=teacher_model,
    config=kd_hidden_config,
)

print(f"Hidden KD Pass applied successfully. Info: {hidden_pass_info}")

### Strategy B: Training

In [ ]:
kd_hidden_trainer = get_trainer(
    model=kd_hidden_model,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
    num_train_epochs=NUM_TRAIN_EPOCHS,
)

print("Starting Logits + Hidden KD training...")
kd_hidden_trainer.train()

### Strategy B: Save Checkpoint

In [ ]:
save_dir_hidden = "saved_models/kd_logits_hidden"
os.makedirs(save_dir_hidden, exist_ok=True)
save_path_hidden = os.path.join(save_dir_hidden, "student_hidden_kd_final.pt")

try:
    torch.save(mg.model.state_dict(), save_path_hidden)
    print(f"Strategy B weights saved to: {save_path_hidden}")
except Exception as e:
    print(f"Failed to save: {e}")

### Strategy B: Load & Evaluate

In [ ]:
print(f"Loading Strategy B weights from: {save_path_hidden}")
raw_state_dict = torch.load(save_path_hidden, map_location="cpu")

try:
    mg.model.load_state_dict(raw_state_dict, strict=True)
    print("Weights loaded successfully (strict mode).")
except RuntimeError:
    print("Strict loading failed; falling back to non-strict mode...")
    mg.model.load_state_dict(raw_state_dict, strict=False)
    print("Weights loaded (non-strict mode).")

mg.model.to("cuda")
mg.model.eval()

eval_trainer = get_trainer(
    model=mg.model,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
)

results_b = eval_trainer.evaluate()

print("\n" + "=" * 50)
print("Comparison Results:")
print("=" * 50)
print(f"  Strategy A (Logits-Only):       0.83988")
print(f"  Strategy B (Logits + Hidden):   {results_b['eval_accuracy']:.5f}")
print("=" * 50)

---
## 8. Strategy C: Logits + Hidden + Attention Joint Distillation

Further extend distillation by aligning attention matrices between student and teacher, in addition to logits and hidden states.

In [ ]:
# Reload teacher on GPU for attention distillation
print(f"Loading teacher model: {TEACHER_CHECKPOINT}")
teacher_model = AutoModelForSequenceClassification.from_pretrained(TEACHER_CHECKPOINT)
teacher_model.eval()
teacher_model.to("cuda")

kd_attn_config = {
    "s_dim": STUDENT_DIM,
    "t_dim": TEACHER_DIM,
    "s_layers": STUDENT_LAYERS,
    "t_layers": TEACHER_LAYERS,
    "alpha_kd": 1.0,       # Logits distillation weight (primary signal)
    "alpha_hidden": 0.1,   # Hidden layer alignment weight (reduced due to large dim gap)
    "alpha_attn": 0.1,     # Attention alignment weight (reduced to prevent overload)
    "temperature": 2.0,
}

print("Applying Logits + Hidden + Attention KD Pass...")
kd_attn_model, attn_pass_info = kd_transform_pass(
    student_graph_or_model=mg,
    teacher_model=teacher_model,
    config=kd_attn_config,
)
print(f"Attention KD Pass applied. Info: {attn_pass_info}")

# Patch state_dict to ensure contiguous tensors (required by some trainer backends)
_orig_state_dict = kd_attn_model.state_dict
def _contiguous_state_dict(*args, **kwargs):
    return {k: v.contiguous() for k, v in _orig_state_dict(*args, **kwargs).items()}
kd_attn_model.state_dict = _contiguous_state_dict

### Strategy C: Training

In [ ]:
print("Starting Logits + Hidden + Attention KD training...")
kd_attn_trainer = get_trainer(
    model=kd_attn_model,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
    num_train_epochs=NUM_TRAIN_EPOCHS,
)
kd_attn_trainer.train()

### Strategy C: Save Checkpoint

In [ ]:
save_dir_attn = "saved_models/kd_logits_hidden_attention"
os.makedirs(save_dir_attn, exist_ok=True)
save_path_attn = os.path.join(save_dir_attn, "student_attention_kd_final.pt")

try:
    torch.save(mg.model.state_dict(), save_path_attn)
    print(f"Strategy C weights saved to: {save_path_attn}")
except Exception as e:
    print(f"Failed to save: {e}")

### Strategy C: Load & Evaluate

In [ ]:
print(f"Loading Strategy C weights from: {save_path_attn}")
state_dict = torch.load(save_path_attn, map_location="cpu")

try:
    mg.model.load_state_dict(state_dict, strict=True)
    print("Weights loaded successfully (strict mode).")
except RuntimeError:
    print("Strict loading failed; attempting key remapping...")
    # Remap pruning parametrization keys back to standard weight keys
    fixed_state_dict = {
        k.replace(".parametrizations.weight.original", ".weight"): v
        for k, v in state_dict.items()
    }
    mg.model.load_state_dict(fixed_state_dict, strict=False)
    print("Weights loaded with key remapping (non-strict mode).")

mg.model.to("cuda")
mg.model.eval()

eval_trainer = get_trainer(
    model=mg.model,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
)

results_c = eval_trainer.evaluate()

print("\n" + "=" * 50)
print("Final Comparison: Attention KD (Strategy C)")
print("=" * 50)
print(f"  Strategy A (Logits-Only):                  0.83988")
print(f"  Strategy C (Logits + Hidden + Attention):   {results_c['eval_accuracy']:.5f}")
print("=" * 50)

---
## 9. Teacher Baseline Evaluation

Evaluate the full-precision `bert-base` teacher model as a reference upper bound.

In [ ]:
print(f"Loading teacher baseline from {TEACHER_CHECKPOINT}...")
reference_base_model = AutoModelForSequenceClassification.from_pretrained(TEACHER_CHECKPOINT)

base_eval_trainer = get_trainer(
    model=reference_base_model,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
)

print("Evaluating the original bert-base teacher model...")
base_eval_results = base_eval_trainer.evaluate()

print("-" * 50)
print(f"[Baseline] bert-base teacher accuracy: {base_eval_results['eval_accuracy']:.5f}")
print("-" * 50)